## Dataset downloading

In [113]:
# # pip install datasets
# from datasets import load_dataset

# from huggingface_hub import login
# login(token="YOUR_TOKEN_HERE")

# # Load the TinyStories dataset
# dataset = load_dataset("roneneldan/TinyStories", split="train")

# # Take the first 5,000 stories
# first_5k = dataset.take(5000)

# # Example: Print the first story
# for item in first_5k.take(1):
#     print(item['text'])

# # Method 1: Save all stories to a single txt file
# with open('tinystories_5k.txt', 'w', encoding='utf-8') as f:
#     for item in first_5k:
#         f.write(item['text'])
#         f.write('\n\n')

In [114]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [115]:
EMBED_DIM = 128
HIDDEN_DIM = 512
NUM_HEAD = 4
BATCH_SIZE = 4
MAX_SEQ_LEN = 100
EPOCH = 100
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PATIENCE = 5
NUM_SAMPLES = 50  # Number of samples to use from the dataset

In [116]:
with open('tinystories_5k.txt', 'r', encoding='utf-8') as f:
    data = f.read()

In [117]:
class Tokenizer:
    def __init__(self, texts):
        self.word2idx = {}
        self.idx2word = {}
        self.build_vocab(texts)
        self.vocab_size = len(self.word2idx)

    def build_vocab(self, texts):
        unique_words = set(word for text in texts for word in text.split())
        self.word2idx = {word: idx + 4 for idx, word in enumerate(unique_words)}  # Start indexing from 4
        self.word2idx['<PAD>'] = 0
        self.word2idx['<SOS>'] = 1
        self.word2idx['<EOS>'] = 2
        self.word2idx['<UNK>'] = 3
        
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}

    def encode(self, text):
        ids = [self.word2idx.get(word, self.word2idx['<UNK>']) for word in text.split()]
        return [self.word2idx['<SOS>']] + ids + [self.word2idx['<EOS>']]

    def decode(self, indices):
        return ' '.join(self.idx2word[idx] for idx in indices)

In [118]:
class SingleHeadAttention(nn.Module):
    def __init__(self, emb_dim, head_dim):
        super().__init__()
        self.query = nn.Linear(emb_dim, head_dim)
        self.key = nn.Linear(emb_dim, head_dim)
        self.value = nn.Linear(emb_dim, head_dim)
        self.d_k = head_dim


    def forward(self, x, context, mask=None):
        Q = self.query(x)
        K = self.key(context)
        V = self.value(context)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask==0, float('-inf'))

        attention_weights = F.softmax(scores, dim=-1)

        output = torch.matmul(attention_weights, V)
        return output


In [119]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads
        self.attention_heads = nn.ModuleList([SingleHeadAttention(emb_dim, self.head_dim) for _ in range(num_heads)])


    def forward(self, x, context, mask=None):
        heads_output = [head(x, context, mask=mask) for head in self.attention_heads]
        concatenated = torch.cat(heads_output, dim=-1)
        return concatenated

In [120]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(emb_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(hidden_dim, emb_dim)
        

    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

In [121]:
class GPTBlock(nn.Module):
    def __init__(self, emb_dim, hidden_dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(emb_dim)
        self.attn = MultiHeadAttention(emb_dim, num_heads)
        self.norm2 = nn.LayerNorm(emb_dim)
        self.ffn = FeedForward(emb_dim, hidden_dim, dropout=0.1)
        self.norm3 = nn.LayerNorm(emb_dim)
        

    def forward(self, x, mask=None):
        x_normed = self.norm1(x)
        attn_output = self.attn(x=x_normed, context=x_normed, mask=mask)
        x = self.norm2(attn_output + x)
        ffn_output = self.ffn(x)
        x = self.norm3(ffn_output + x)
        return x

In [122]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, max_seq_length, num_heads, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_seq_length, emb_dim))
        self.gpt_blocks = nn.ModuleList([GPTBlock(emb_dim, hidden_dim, num_heads) for _ in range(num_layers)])
        self.output_layer = nn.Linear(emb_dim, vocab_size)


    def forward(self, x, mask=None):
        seq_len = x.size(1)
        x = self.embedding(x) + self.positional_encoding[:, :seq_len, :]
        for block in self.gpt_blocks:
            x = block(x, mask=mask)

        logits = self.output_layer(x)
        return logits


In [123]:
def create_causal_mask(seq_len):
    mask = torch.tril(torch.ones(seq_len, seq_len)).bool()  # lower triangular, bool
    return mask.unsqueeze(0)  # (1, seq_len, seq_len)


def create_gpt_mask(x, pad_idx=0):
    seq_len = x.size(1)
    causal_mask = create_causal_mask(seq_len).to(x.device)
    pad_mask = (x != pad_idx).unsqueeze(1)
    combined_mask = causal_mask & pad_mask
    return combined_mask

In [124]:
all_texts = data.split('\n\n')  # Split stories by double newlines

In [125]:
texts = all_texts[:NUM_SAMPLES]  # Use only the first 5,000 stories for now

In [126]:
class TinyStoriesDataset(Dataset):
    def __init__(self, texts, tokenizer, max_seq_len):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        token_ids = self.tokenizer.encode(text)
        # if len(token_ids) > self.max_seq_len:
        #     token_ids = token_ids[:self.max_seq_len-2]
        #     token_ids = [self.tokenizer.word2idx['<SOS>']] + token_ids + [self.tokenizer.word2idx['<EOS>']]
        # else:
        #     token_ids += [self.tokenizer.word2idx['<PAD>']] * (self.max_seq_len - len(token_ids))

        if len(token_ids) >= self.max_seq_len-2:
            token_ids = token_ids[:self.max_seq_len-2]
            token_ids = [self.tokenizer.word2idx['<SOS>']] + token_ids + [self.tokenizer.word2idx['<EOS>']]
        else:
            token_ids = [self.tokenizer.word2idx['<SOS>']] + token_ids + [self.tokenizer.word2idx['<EOS>']]
            # print(f"current token id {len(token_ids)}")
            need_padding = self.max_seq_len - len(token_ids)
            # print(f"need padding length {need_padding}")
            token_ids += [self.tokenizer.word2idx['<PAD>']] * need_padding

        
        return torch.tensor(token_ids, dtype=torch.long)

In [127]:
tokenizer = Tokenizer(texts)
print(f"Vocabulary size: {tokenizer.vocab_size}")
dataset = TinyStoriesDataset(texts, tokenizer, MAX_SEQ_LEN)
len(dataset)

Vocabulary size: 627


50

In [128]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
print(f"Total: {len(dataset)}, Train: {len(train_dataset)}, Val: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

Total: 50, Train: 40, Val: 10


In [129]:
model = GPTLanguageModel(vocab_size=tokenizer.vocab_size, emb_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, max_seq_length=MAX_SEQ_LEN, num_heads=NUM_HEAD, num_layers=2).to(DEVICE)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.word2idx['<PAD>'])
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

best_val_loss = float('inf')
best_val_loss = float('inf')

patiece = 0

In [130]:
model.train()
for epoch in range(EPOCH):
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(DEVICE)
        mask = create_gpt_mask(batch, pad_idx=tokenizer.word2idx['<PAD>'])
        output = model(batch, mask=mask)
        
        loss = criterion(output.view(-1, tokenizer.vocab_size), batch.view(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for val_batch in val_loader:
            val_batch = val_batch.to(DEVICE)
            mask = create_gpt_mask(val_batch, pad_idx=tokenizer.word2idx['<PAD>'])
            val_logits = model(val_batch, mask=mask)
            loss = criterion(val_logits.view(-1, tokenizer.vocab_size), val_batch.view(-1))
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{EPOCH}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

    # Early stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patiece = 0
        torch.save(model.state_dict(), 'best_model.pth')
    else:
        patiece += 1
        if patiece >= PATIENCE:
            print("Early stopping triggered.")
            break

    model.train()

Epoch 1/100, Train Loss: 5.4242, Val Loss: 4.4269
Epoch 2/100, Train Loss: 3.6993, Val Loss: 3.4083
Epoch 3/100, Train Loss: 2.6634, Val Loss: 2.7687
Epoch 4/100, Train Loss: 1.9485, Val Loss: 2.3681
Epoch 5/100, Train Loss: 1.4689, Val Loss: 2.1060
Epoch 6/100, Train Loss: 1.1002, Val Loss: 1.9196
Epoch 7/100, Train Loss: 0.8476, Val Loss: 1.7920
Epoch 8/100, Train Loss: 0.6481, Val Loss: 1.7097
Epoch 9/100, Train Loss: 0.4959, Val Loss: 1.6269
Epoch 10/100, Train Loss: 0.3729, Val Loss: 1.5680
Epoch 11/100, Train Loss: 0.2796, Val Loss: 1.5319
Epoch 12/100, Train Loss: 0.2122, Val Loss: 1.4893
Epoch 13/100, Train Loss: 0.1592, Val Loss: 1.4799
Epoch 14/100, Train Loss: 0.1245, Val Loss: 1.4570
Epoch 15/100, Train Loss: 0.0976, Val Loss: 1.4462
Epoch 16/100, Train Loss: 0.0802, Val Loss: 1.4450
Epoch 17/100, Train Loss: 0.0668, Val Loss: 1.4346
Epoch 18/100, Train Loss: 0.0571, Val Loss: 1.4316
Epoch 19/100, Train Loss: 0.0494, Val Loss: 1.4304
Epoch 20/100, Train Loss: 0.0439, Val Lo

In [131]:
def generate_text(model, tokenizer, prompt, max_length=50):
    model.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(DEVICE)
    generated_ids = input_ids

    with torch.no_grad():
        for _ in range(max_length):
            mask = create_gpt_mask(generated_ids, pad_idx=tokenizer.word2idx['<PAD>'])
            logits = model(generated_ids, mask=mask)
            next_token_logits = logits[:, -1, :]
            next_token_id = torch.argmax(next_token_logits, dim=-1).unsqueeze(0)
            generated_ids = torch.cat((generated_ids, next_token_id), dim=1)

            if next_token_id.item() == tokenizer.word2idx['<EOS>']:
                break

    generated_text = tokenizer.decode(generated_ids.squeeze().tolist())
    return generated_text

In [132]:
generate_text(model, tokenizer, "One day, a little girl", max_length=100)

'<SOS> One day, a little girl <EOS> <EOS>'

In [133]:
input_ids = torch.tensor(tokenizer.encode("One day, a little girl"), dtype=torch.float32).unsqueeze(0).to(DEVICE)
create_gpt_mask(input_ids)

tensor([[[ True, False, False, False, False, False, False],
         [ True,  True, False, False, False, False, False],
         [ True,  True,  True, False, False, False, False],
         [ True,  True,  True,  True, False, False, False],
         [ True,  True,  True,  True,  True, False, False],
         [ True,  True,  True,  True,  True,  True, False],
         [ True,  True,  True,  True,  True,  True,  True]]])